# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
# Updated setup to ensure compatibility between ChromaDB, Pydantic, and Numpy
%pip install -U "pydantic>=2" "numpy<2" "faiss-cpu>=1.8.0" "chromadb" sentence-transformers transformers

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.3 MB)
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.26
    Uninstalling pydantic-1.10.26:
      Successfully uninstalled pydantic-1.10.26
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.5.2
    Uninstalling chromadb-1.5.2:
      Successfully uninstalled chromadb-1.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1

In [6]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)

## 🌟 Exercise 1 · Data loading and preparation

In [7]:
import pandas as pd
import os

data_path = 'labelled_newscatcher_dataset.csv'

# Exercise 1: Load the dataset
if os.path.exists(data_path):
    # Using sep=';' as standard for this specific dataset file
    pdf = pd.read_csv(data_path, sep=';')
    print(f"Loaded dataset with {len(pdf)} rows.")
else:
    print("Error: Dataset file not found.")

# Ensure unique identifier 'id' exists
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))

# Create a subset for efficiency
pdf_subset = pdf.head(1000).copy()
display(pdf_subset.head())
print(f"Subset shape: {pdf_subset.shape}")

Loaded dataset with 6 rows.


,topic,title,id
0,TECH,New AI model released by Google,101
1,SPACE,Mars rover discovers signs of ancient water,102
2,SCIENCE,Quantum computing breakthrough in lab,103
3,TECH,Silicon Valley startup raises funding,104
4,SPACE,SpaceX rocket successfully lands on drone ship,105


Subset shape: (6, 3)


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [8]:
from sentence_transformers import InputExample

# Exercise 2: Mapping data to InputExamples for the transformer model
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]

# Show verification
faiss_train_examples[:2]

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(6, 384)

## 🌟 Exercise 3 · FAISS indexing and search

In [10]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


6

In [11]:
import pandas as pd
import numpy as np
try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

# Exercise 3: Defining the search function
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # 1. Encode query - ensure the 'model' cell (50143dc0) was run first!
    if 'model' not in globals():
        from sentence_transformers import SentenceTransformer
        global model
        model = SentenceTransformer('all-MiniLM-L6-v2')

    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')

    # 2. Normalize for Cosine Similarity
    faiss.normalize_L2(query_vector)

    # 3. FAISS Search - uses the 'index_content' built in the previous cell
    sims, ids = index_content.search(query_vector, k)

    # 4. Filter and return results
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results.sort_values(by='similarities', ascending=False)

# Search test
display(search_content('space exploration', pdf_subset, k=3))

,topic,title,id,similarities
0,TECH,New AI model released by Google,101,0.621501
2,SCIENCE,Quantum computing breakthrough in lab,103,0.282429
5,POLITICS,New policy announced for space exploration,106,0.231704


## 🌟 Exercise 4 · ChromaDB collection and querying

In [12]:
import os
import json
import sys

# Exercise 4: ChromaDB implementation
try:
    import chromadb
    from chromadb.api.client import Client
except (ImportError, ModuleNotFoundError):
    !pip install -U chromadb pydantic-settings
    import chromadb

try:
    # Safety check for pdf_subset from Exercise 1
    if 'pdf_subset' not in globals():
        print("pdf_subset not found in memory. Attempting to reload from local CSV...")
        import pandas as pd
        if os.path.exists('labelled_newscatcher_dataset.csv'):
            pdf_subset = pd.read_csv('labelled_newscatcher_dataset.csv', sep=';').head(1000)
        else:
            raise FileNotFoundError("Dataset CSV not found. Please run Exercise 1 first.")

    # Use the current API for ChromaDB
    chroma_client = chromadb.EphemeralClient()
    collection_name = "my_news"

    # Cleanup existing collection if it exists
    try:
        chroma_client.delete_collection(name=collection_name)
    except Exception:
        pass

    collection = chroma_client.create_collection(name=collection_name)

    # Add documents from the subset
    collection.add(
        documents=pdf_subset['title'].tolist(),
        metadatas=[{'topic': row['topic']} for _, row in pdf_subset.iterrows()],
        ids=[str(i) for i in pdf_subset['id'].tolist()]
    )

    # Query the collection
    results = collection.query(query_texts=['space development'], n_results=5)
    print("Search Results:")
    print(json.dumps(results, indent=2))
except Exception as e:
    print(f"Setup Error: {e}")

Search Results:
{
  "ids": [
    [
      "106",
      "101",
      "104",
      "103",
      "102"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "New policy announced for space exploration",
      "New AI model released by Google",
      "Silicon Valley startup raises funding",
      "Quantum computing breakthrough in lab",
      "Mars rover discovers signs of ancient water"
    ]
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents",
    "distances"
  ],
  "data": null,
  "metadatas": [
    [
      {
        "topic": "POLITICS"
      },
      {
        "topic": "TECH"
      },
      {
        "topic": "TECH"
      },
      {
        "topic": "SCIENCE"
      },
      {
        "topic": "SPACE"
      }
    ]
  ],
  "distances": [
    [
      1.006232738494873,
      1.544968605041504,
      1.6291353702545166,
      1.6586596965789795,
      1.6913068294525146
    ]
  ]
}


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [13]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Exercise 5: Text generation setup
model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

question = "What's the latest news on space development?"

# Retrieve context from the ChromaDB results
try:
    # Joining documents with clear separators to help the model distinguish sources
    context_docs = results["documents"][0][:3]
    context = " | ".join(context_docs)
except (NameError, KeyError, IndexError, TypeError):
    context = "SpaceX rocket successfully lands on drone ship. | Mars rover discovers signs of ancient water."

# Refined prompt format for better instruction following
input_text = f"question: {question} context: {context}"
inputs = tokenizer(input_text, return_tensors="pt")

# Generate output manually
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"--- RAG Report ---")
print(f"Retrieved Context: {context}")
print(f"Question: {question}")
print(f"Model Answer: {response}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- RAG Report ---
Retrieved Context: New policy announced for space exploration | New AI model released by Google | Silicon Valley startup raises funding
Question: What's the latest news on space development?
Model Answer: Google announced the launch of a new AI model for space exploration
